### environment & Data Validation

In [ ]:

# ============================================================
# UNGDC PROJECT
# Notebook 01: Data Preparation
# Step 01: Environment and File Validation
# ============================================================

from pathlib import Path
import sys
import platform
import pandas as pd
import numpy as np
import pyarrow as pa

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_DIR = Path.home() / "Projects" / "7-masoud"

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

ARCHIVE_PATH = RAW_DIR / "UNGDC_1946-2025.tar.gz"
SPEAKERS_PATH = RAW_DIR / "Speakers_by_session.xlsx"
README_PATH = RAW_DIR / "README.txt"

# ------------------------------------------------------------
# 2. Environment information
# ------------------------------------------------------------

print("=" * 60)
print("UNGDC - ENVIRONMENT VALIDATION")
print("=" * 60)

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"PyArrow: {pa.__version__}")

# ------------------------------------------------------------
# 3. Validate project paths
# ------------------------------------------------------------

assert PROJECT_DIR.is_dir(), "Project directory not found"
assert RAW_DIR.is_dir(), "Raw data directory not found"
assert PROCESSED_DIR.is_dir(), "Processed directory not found"

required_files = {
    "UNGDC Archive": ARCHIVE_PATH,
    "Speakers Metadata": SPEAKERS_PATH,
    "Dataset README": README_PATH,
}

for name, path in required_files.items():
    assert path.is_file(), f"Missing file: {path}"

    size_mb = path.stat().st_size / (1024 ** 2)

    print(f"[OK] {name}: {size_mb:.2f} MB")

print("=" * 60)
print("STEP 01: ALL CHECKS PASSED")



## 2. Archive Structure and Metadata Audit

### Objective

This step examines the internal structure of the United Nations General Debate Corpus (UNGDC), covering 1946–2025, before extracting or processing speech texts.

### Validation procedures

1. Inspect the archive without extracting files.
2. Identify speech files using the documented filename convention.
3. Extract country/entity codes, UN session numbers, and years.
4. Identify non-standard files and unexpected naming patterns.
5. Check for duplicate speech identifiers.
6. Verify the consistency between session numbers and calendar years.
7. Summarize the temporal and geographical coverage.

### Expected output

A speech-level metadata table containing:

- `country_code`: ISO-style country or entity identifier.
- `session`: United Nations General Assembly session number.
- `year`: Calendar year of the speech.
- `filename`: Original speech filename.
- `archive_path`: Internal location in the compressed archive.
- `size_bytes`: Uncompressed file size in bytes.

### Methodological considerations

A valid filename does not guarantee valid textual content. Text decoding, completeness, duplication, and linguistic quality will be assessed separately.

The European Union may appear as a non-state entity and must be distinguished from sovereign states in subsequent analyses.

No speech texts are modified during this step.


In [ ]:

# ============================================================
# STEP 02: ARCHIVE STRUCTURE AND METADATA AUDIT
# ============================================================

import re
import tarfile
from collections import Counter

# Accept two- or three-letter entity identifiers.
# Example: IRN_01_1946.txt, USA_80_2025.txt
speech_pattern = re.compile(
    r"^(?P<country_code>[A-Z]{2,3})_"
    r"(?P<session>\d{1,2})_"
    r"(?P<year>\d{4})\.txt$"
)

records = []
unrecognized_files = []
archive_members = []

with tarfile.open(ARCHIVE_PATH, mode="r:gz") as archive:

    for member in archive:
        if not member.isfile():
            continue

        archive_members.append(member.name)

        filename = member.name.rsplit("/", 1)[-1]
        match = speech_pattern.fullmatch(filename)

        if match is None:
            unrecognized_files.append(member.name)
            continue

        metadata = match.groupdict()

        records.append({
            "country_code": metadata["country_code"],
            "session": int(metadata["session"]),
            "year": int(metadata["year"]),
            "filename": filename,
            "archive_path": member.name,
            "size_bytes": member.size
        })

# Build a speech-level metadata table.
speech_index = pd.DataFrame.from_records(records)

# Validate required metadata columns.
required_columns = [
    "country_code",
    "session",
    "year",
    "filename",
    "archive_path",
    "size_bytes"
]

assert all(
    column in speech_index.columns
    for column in required_columns
), "Required metadata columns are missing."

# ------------------------------------------------------------
# Structural checks
# ------------------------------------------------------------

duplicate_filenames = speech_index[
    speech_index.duplicated(
        subset=["filename"], keep=False
    )
]

duplicate_country_year = speech_index[
    speech_index.duplicated(
        subset=["country_code", "year"], keep=False
    )
]

invalid_sessions = speech_index[
    speech_index["session"] !=
    speech_index["year"] - 1945
]

invalid_years = speech_index[
    ~speech_index["year"].between(1946, 2025)
]

empty_files = speech_index[
    speech_index["size_bytes"] == 0
]

# ------------------------------------------------------------
# Archive summary
# ------------------------------------------------------------

print("=" * 65)
print("UNGDC ARCHIVE STRUCTURE AUDIT")
print("=" * 65)

print(f"Total regular files: {len(archive_members):,}")
print(f"Recognized speech files: {len(speech_index):,}")
print(f"Unrecognized files: {len(unrecognized_files):,}")

print(
    "Unique countries/entities:",
    speech_index["country_code"].nunique()
)

print(
    "Temporal coverage:",
    speech_index["year"].min(),
    "-",
    speech_index["year"].max()
)

print(
    "Number of represented years:",
    speech_index["year"].nunique()
)

print("-" * 65)
print("STRUCTURAL VALIDATION")
print("-" * 65)

print("Duplicate filenames:", len(duplicate_filenames))
print("Duplicate country-year records:", len(duplicate_country_year))
print("Invalid session-year pairs:", len(invalid_sessions))
print("Out-of-range years:", len(invalid_years))
print("Zero-byte speech files:", len(empty_files))

print("-" * 65)
print("UNRECOGNIZED FILES")
print("-" * 65)

for name in unrecognized_files[:20]:
    print(name)

if len(unrecognized_files) > 20:
    print(
        f"... and {len(unrecognized_files) - 20} more."
    )

print("-" * 65)
print("SAMPLE METADATA")
print("-" * 65)

display(
    speech_index.sort_values(
        ["year", "country_code"]
    ).head(10)
)



### 2.1. Classification of Non-Speech Archive Members

The initial archive inspection identified 11,141 files matching the documented UNGDC speech filename convention, alongside additional non-matching files.

Before excluding these files, we classify them into:

- **AppleDouble metadata:** Files beginning with `._`, commonly associated with macOS filesystem metadata.
- **Other hidden files:** Hidden files not classified as AppleDouble metadata.
- **Other non-speech files:** Files that do not match the speech naming convention and require manual review.

We also validate the directory structure of recognized speeches against their session and year identifiers.

**Quality assurance principle:** Files must not be excluded from the analytical corpus solely because their names do not match the expected pattern. Exclusion decisions must be supported by an explicit classification and documented rationale.


In [ ]:

# ============================================================
# STEP 02.1: ARCHIVE ARTIFACT CLASSIFICATION
# ============================================================

from collections import Counter
from pathlib import PurePosixPath

# ------------------------------------------------------------
# 1. Classify non-matching regular files
# ------------------------------------------------------------

artifact_records = []

for archive_path in unrecognized_files:

    filename = PurePosixPath(archive_path).name

    if filename.startswith("._"):
        category = "AppleDouble metadata"

    elif filename.startswith("."):
        category = "Other hidden file"

    else:
        category = "Other non-speech file"

    artifact_records.append({
        "archive_path": archive_path,
        "filename": filename,
        "category": category
    })

artifacts_df = pd.DataFrame(
    artifact_records,
    columns=["archive_path", "filename", "category"]
)

category_counts = Counter(
    record["category"] for record in artifact_records
)

print("=" * 65)
print("ARCHIVE ARTIFACT CLASSIFICATION")
print("=" * 65)

for category, count in category_counts.most_common():
    print(f"{category}: {count:,}")

print(f"\nTotal classified: {len(artifacts_df):,}")

assert len(artifacts_df) == len(unrecognized_files)
assert len(speech_index) + len(artifacts_df) == len(archive_members)

# ------------------------------------------------------------
# 2. Inspect files requiring manual review
# ------------------------------------------------------------

review_required = artifacts_df[
    artifacts_df["category"] != "AppleDouble metadata"
]

print("\nFiles requiring further review:")
print(len(review_required))

display(review_required.head(30))

# ------------------------------------------------------------
# 3. Validate speech directory conventions
# ------------------------------------------------------------

def expected_archive_path(row):
    return (
        f"TXT/Session {row['session']:02d} - "
        f"{row['year']}/{row['filename']}"
    )

expected_paths = speech_index.apply(
    expected_archive_path,
    axis=1
)

path_mismatch = speech_index.loc[
    speech_index["archive_path"] != expected_paths
].copy()

print("\nSpeech path mismatches:", len(path_mismatch))

display(path_mismatch.head(20))


# ------------------------------------------------------------
# 4. Final classification and structural quality assessment
# ------------------------------------------------------------

known_system_files = {
    ".DS_Store",
    ".DS_Store-to-UTF-8.txt",
    ".Rhistory",
    ".Rapp.history",
}

# Only explicitly recognized system artifacts are excluded.
known_artifacts = artifacts_df[
    (artifacts_df["category"] == "AppleDouble metadata")
    | (artifacts_df["filename"].isin(known_system_files))
].copy()

unexpected_artifacts = artifacts_df[
    ~artifacts_df.index.isin(known_artifacts.index)
].copy()

checks = {
    "No duplicate filenames":
        duplicate_filenames.empty,

    "No duplicate country-year IDs":
        duplicate_country_year.empty,

    "Valid session-year mapping":
        invalid_sessions.empty,

    "Years within expected range":
        invalid_years.empty,

    "No zero-byte speech files":
        empty_files.empty,

    "Archive member counts reconcile":
        len(speech_index) + len(artifacts_df)
        == len(archive_members),

    "No unexplained archive artifacts":
        unexpected_artifacts.empty,

    "Speech paths match convention":
        path_mismatch.empty
}

print("=" * 65)
print("FINAL STRUCTURAL QUALITY ASSESSMENT")
print("=" * 65)

print(f"Recognized speeches: {len(speech_index):,}")
print(f"Known system artifacts: {len(known_artifacts):,}")
print(f"Unexplained artifacts: {len(unexpected_artifacts):,}")

print("-" * 65)

for name, passed in checks.items():
    status = "PASS" if passed else "REVIEW"
    print(f"[{status}] {name}")

structural_audit_passed = all(checks.values())

print("-" * 65)
print("Structural audit passed:", structural_audit_passed)

if not unexpected_artifacts.empty:
    display(unexpected_artifacts)

assert structural_audit_passed, (
    "Structural validation failed. Review flagged items."
)



## 3. Text Integrity and Quality Assessment

### Objective

This section evaluates the textual integrity of all recognized speeches before preprocessing, feature extraction, or statistical modeling.

### Quality dimensions

1. **Decoding integrity:** Verify that speech files can be decoded using UTF-8.
2. **Text completeness:** Detect empty or whitespace-only documents.
3. **Document length:** Calculate character counts, word counts, and line counts.
4. **Encoding anomalies:** Identify replacement characters and unexpected control characters.
5. **Exact duplicates:** Detect identical raw document contents using SHA-256 hashes.
6. **Length anomalies:** Flag potentially unusual document lengths for subsequent investigation.

### Measurement considerations

Word counts are based on Unicode-aware regular-expression tokenization and represent descriptive token counts rather than linguistically validated word segmentation.

Unusually short or long speeches are not automatically considered invalid. Length thresholds are diagnostic flags, not exclusion criteria.

Exact file hashes detect byte-identical documents but do not detect near-duplicates, translated duplicates, or semantically equivalent texts.

### Data preservation

All original speech files remain unchanged. No stopword removal, stemming, lemmatization, case normalization, or document exclusion is performed at this stage.

The audit generates document-level quality indicators for further review before constructing the analytical dataset.


In [ ]:

# ============================================================
# STEP 03: TEXT INTEGRITY AND QUALITY ASSESSMENT
# ============================================================

import hashlib
import tarfile
import re

from collections import Counter

# ------------------------------------------------------------
# 1. Define diagnostic patterns
# ------------------------------------------------------------

word_pattern = re.compile(r"\b\w+\b", flags=re.UNICODE)

control_pattern = re.compile(
    r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]"
)

quality_records = []
decoding_errors = []

# ------------------------------------------------------------
# 2. Read speech contents directly from the archive
# ------------------------------------------------------------

with tarfile.open(ARCHIVE_PATH, mode="r:gz") as archive:

    members = {
        member.name: member
        for member in archive.getmembers()
        if member.isfile()
    }

    for row in speech_index.itertuples(index=False):

        member = members.get(row.archive_path)

        if member is None:
            raise FileNotFoundError(row.archive_path)

        extracted = archive.extractfile(member)

        if extracted is None:
            raise RuntimeError(
                f"Cannot read: {row.archive_path}"
            )

        raw_bytes = extracted.read()

        # Hash the original bytes, without modifying the text.
        content_hash = hashlib.sha256(
            raw_bytes
        ).hexdigest()

        # Decode strictly to avoid hiding encoding errors.
        try:
            text = raw_bytes.decode("utf-8-sig")
            decoding_success = True

        except UnicodeDecodeError as exc:
            decoding_errors.append({
                "archive_path": row.archive_path,
                "error": str(exc)
            })

            text = None
            decoding_success = False

        if decoding_success:

            word_count = len(
                word_pattern.findall(text)
            )

            character_count = len(text)

            line_count = len(text.splitlines())

            whitespace_only = not bool(text.strip())

            replacement_count = text.count("\ufffd")

            control_count = len(
                control_pattern.findall(text)
            )

        else:
            word_count = np.nan
            character_count = np.nan
            line_count = np.nan
            whitespace_only = pd.NA
            replacement_count = np.nan
            control_count = np.nan

        quality_records.append({
            "archive_path": row.archive_path,
            "sha256": content_hash,
            "actual_size_bytes": len(raw_bytes),
            "utf8_valid": decoding_success,
            "character_count": character_count,
            "word_count": word_count,
            "line_count": line_count,
            "whitespace_only": whitespace_only,
            "replacement_characters": replacement_count,
            "control_characters": control_count
        })

# ------------------------------------------------------------
# 3. Merge quality indicators with archive metadata
# ------------------------------------------------------------

quality_df = pd.DataFrame(quality_records)

speech_quality = speech_index.merge(
    quality_df,
    on="archive_path",
    how="left",
    validate="one_to_one",
    indicator=True
)

assert len(speech_quality) == len(speech_index)

assert (
    speech_quality["_merge"] == "both"
).all(), "Some speech records were not matched."

speech_quality = speech_quality.drop(
    columns="_merge"
)

# ------------------------------------------------------------
# 4. Verify size consistency
# ------------------------------------------------------------

size_mismatches = speech_quality[
    speech_quality["size_bytes"]
    != speech_quality["actual_size_bytes"]
]

# ------------------------------------------------------------
# 5. Report initial quality results
# ------------------------------------------------------------

print("=" * 65)
print("TEXT INTEGRITY AUDIT")
print("=" * 65)

print(f"Speeches inspected: {len(speech_quality):,}")

print(
    "UTF-8 decoding failures:",
    len(decoding_errors)
)

print(
    "Archive size mismatches:",
    len(size_mismatches)
)

print(
    "Whitespace-only speeches:",
    int(speech_quality["whitespace_only"].eq(True).sum())
)

print(
    "Speeches with replacement characters:",
    int(
        speech_quality["replacement_characters"]
        .gt(0).sum()
    )
)

print(
    "Speeches with control characters:",
    int(
        speech_quality["control_characters"]
        .gt(0).sum()
    )
)

print("-" * 65)

if decoding_errors:
    display(pd.DataFrame(decoding_errors).head(20))

display(speech_quality.head())



### 3.1. Document Length and Exact Duplicate Diagnostics

This analysis examines document length distributions and identifies exact duplicate speech contents across the UNGDC corpus.

The following indicators are evaluated:

- Word count distribution and selected percentiles.
- Potentially short and long documents.
- Exact duplicate content based on SHA-256 hashes.
- The shortest and longest speeches in the corpus.

Thresholds of 100 and 15,000 words are used as preliminary diagnostic flags, not as automatic exclusion criteria.

Exact duplicate detection is based on byte-level identity. Near-duplicate and semantic similarity assessments will be conducted separately if required.

No speeches are removed or modified during this analysis.


In [ ]:

# ============================================================
# STEP 03.1: DOCUMENT LENGTH AND DUPLICATE DIAGNOSTICS
# ============================================================

# 1. Identify exact duplicate documents
duplicate_content = speech_quality[
    speech_quality.duplicated(
        subset=["sha256"],
        keep=False
    )
].sort_values("sha256")

# 2. Flag potentially unusual document lengths
MIN_WORDS_FLAG = 100
MAX_WORDS_FLAG = 15000

speech_quality["flag_short"] = (
    speech_quality["word_count"] < MIN_WORDS_FLAG
)

speech_quality["flag_long"] = (
    speech_quality["word_count"] > MAX_WORDS_FLAG
)

# 3. Summarize word count distribution
print("=" * 65)
print("DOCUMENT LENGTH AND DUPLICATE DIAGNOSTICS")
print("=" * 65)

print("\nWord count distribution:")

print(
    speech_quality["word_count"].describe(
        percentiles=[
            0.01, 0.05, 0.25,
            0.50, 0.75, 0.95, 0.99
        ]
    ).round(2)
)

print("\nDiagnostic flags:")

print(
    "Speeches below 100 words:",
    int(speech_quality["flag_short"].sum())
)

print(
    "Speeches above 15,000 words:",
    int(speech_quality["flag_long"].sum())
)

print(
    "Documents involved in exact duplicates:",
    len(duplicate_content)
)

print(
    "Duplicate hash groups:",
    duplicate_content["sha256"].nunique()
)

# 4. Inspect shortest documents
print("\nShortest speeches:")

display(
    speech_quality[
        [
            "country_code",
            "year",
            "word_count",
            "character_count",
            "filename"
        ]
    ].sort_values("word_count").head(15)
)

# 5. Inspect longest documents
print("\nLongest speeches:")

display(
    speech_quality[
        [
            "country_code",
            "year",
            "word_count",
            "character_count",
            "filename"
        ]
    ].sort_values(
        "word_count",
        ascending=False
    ).head(15)
)

# 6. Inspect duplicate content
print("\nExact duplicate documents:")

display(
    duplicate_content[
        [
            "country_code",
            "year",
            "filename",
            "word_count",
            "sha256"
        ]
    ].head(30)
)



### 3.2. Investigation of Textual Anomalies

#### Objective

This analysis characterizes unexpected Unicode replacement characters and control characters identified during the initial text integrity audit.

#### Procedures

1. Identify documents containing potentially problematic characters.
2. Calculate the frequency of individual anomalous characters.
3. Examine short textual contexts around selected anomalies.
4. Assess the distribution of affected documents across years and countries.

#### Methodological considerations

The presence of control characters does not necessarily indicate corrupted text. Some characters may reflect document conversion or formatting artifacts.

Character-level anomalies will be reviewed before defining any cleaning rules. Replacement characters will not be automatically removed because they may indicate irrecoverable information loss in the original text.

All inspections are performed on unmodified source documents.

#### Decision rule

No automatic exclusion or text modification will be performed until the type, frequency, and potential analytical impact of each anomaly have been evaluated.


In [ ]:

# ============================================================
# STEP 03.2: TEXTUAL ANOMALY INVESTIGATION
# ============================================================

import unicodedata
from collections import Counter

# ------------------------------------------------------------
# 1. Identify affected speeches
# ------------------------------------------------------------

anomaly_mask = (
    speech_quality["replacement_characters"].gt(0)
    | speech_quality["control_characters"].gt(0)
)

affected_speeches = speech_quality.loc[
    anomaly_mask
].copy()

print("Affected speeches:", len(affected_speeches))

# ------------------------------------------------------------
# 2. Character-level inspection
# ------------------------------------------------------------

character_counts = Counter()
character_documents = Counter()
context_samples = []

MAX_SAMPLES = 20
CONTEXT_WIDTH = 45

with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:

    for row in affected_speeches.itertuples(index=False):

        extracted = archive.extractfile(row.archive_path)

        if extracted is None:
            raise RuntimeError(
                f"Unable to read {row.archive_path}"
            )

        text = extracted.read().decode("utf-8-sig")

        matches = list(control_pattern.finditer(text))
        matches += list(re.finditer("\ufffd", text))
        matches.sort(key=lambda match: match.start())

        document_characters = set()

        for match in matches:

            character = match.group()
            character_counts[character] += 1
            document_characters.add(character)

            if len(context_samples) < MAX_SAMPLES:
                start = max(0, match.start() - CONTEXT_WIDTH)
                end = min(
                    len(text),
                    match.end() + CONTEXT_WIDTH
                )

                context_samples.append({
                    "country_code": row.country_code,
                    "year": row.year,
                    "filename": row.filename,
                    "character": repr(character),
                    "context": repr(text[start:end])
                })

        for character in document_characters:
            character_documents[character] += 1

# ------------------------------------------------------------
# 3. Character frequency summary
# ------------------------------------------------------------

character_summary = pd.DataFrame([
    {
        "character": repr(char),
        "unicode_code": f"U+{ord(char):04X}",
        "unicode_name": unicodedata.name(
            char, "UNNAMED CONTROL CHARACTER"
        ),
        "total_occurrences": count,
        "affected_documents": character_documents[char]
    }
    for char, count in character_counts.items()
]).sort_values(
    "total_occurrences",
    ascending=False
)

print("\nAnomalous character frequencies:")
display(character_summary)

# ------------------------------------------------------------
# 4. Contextual examples
# ------------------------------------------------------------

print("\nSelected anomaly contexts:")
display(pd.DataFrame(context_samples))

# ------------------------------------------------------------
# 5. Temporal distribution
# ------------------------------------------------------------

annual_anomalies = (
    affected_speeches
    .groupby("year")
    .size()
    .rename("affected_speeches")
    .reset_index()
)

annual_total = (
    speech_quality
    .groupby("year")
    .size()
    .rename("total_speeches")
    .reset_index()
)

annual_anomalies = annual_total.merge(
    annual_anomalies,
    on="year",
    how="left",
    validate="one_to_one"
)

annual_anomalies["affected_speeches"] = (
    annual_anomalies["affected_speeches"]
    .fillna(0)
    .astype(int)
)

annual_anomalies["affected_percentage"] = (
    100
    * annual_anomalies["affected_speeches"]
    / annual_anomalies["total_speeches"]
)

print("\nYears with highest anomaly prevalence:")

display(
    annual_anomalies.sort_values(
        "affected_percentage",
        ascending=False
    ).head(15)
)

# ------------------------------------------------------------
# 6. Country-level summary
# ------------------------------------------------------------

print("\nCountries with most affected speeches:")

display(
    affected_speeches.groupby("country_code")
    .size()
    .rename("affected_speeches")
    .sort_values(ascending=False)
    .head(15)
)

print("\nIran, United States, and China:")

display(
    affected_speeches.loc[
        affected_speeches["country_code"].isin(
            ["IRN", "USA", "CHN"]
        ),
        [
            "country_code",
            "year",
            "filename",
            "replacement_characters",
            "control_characters"
        ]
    ].sort_values(["country_code", "year"])
)



### 3.3. Controlled Text Normalization

#### Objective

This step implements a minimal, explicitly documented correction for form-feed characters (U+000C) identified during the textual integrity audit.

#### Cleaning policy

- Preserve the original UTF-8 source files.
- Replace form-feed characters with a single space.
- Preserve punctuation, capitalization, word order, and document content.
- Retain Unicode replacement characters (U+FFFD) for further investigation.
- Do not apply tokenization, stemming, lemmatization, or stopword removal.

#### Validation

The cleaning procedure must verify that:

1. All form-feed characters are removed.
2. No additional content is unintentionally modified.
3. Document identifiers remain unchanged.
4. Cleaned text can be reproduced deterministically.

This normalization addresses formatting artifacts rather than substantive linguistic content.

Changes in descriptive token counts will be examined before the cleaned corpus is approved.


In [ ]:

# ============================================================
# STEP 03.3: CONTROLLED TEXT NORMALIZATION
# ============================================================

import tarfile
import hashlib

# ------------------------------------------------------------
# 1. Define a minimal normalization function
# ------------------------------------------------------------

def normalize_text(text: str) -> str:
    """Replace form-feed characters while preserving other text."""
    return text.replace("\x0c", " ")


# ------------------------------------------------------------
# 2. Validate the normalization function
# ------------------------------------------------------------

test_cases = {
    "alpha\x0cbeta": "alpha beta",
    "alpha \x0c beta": "alpha   beta",
    "alpha\nbeta": "alpha\nbeta",
    "alpha\ufffdbeta": "alpha\ufffdbeta",
}

for original, expected in test_cases.items():
    assert normalize_text(original) == expected

print("Normalization unit tests: PASSED")


# ------------------------------------------------------------
# 3. Examine all documents without changing source files
# ------------------------------------------------------------

normalization_records = []

with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:

    for row in speech_index.itertuples(index=False):

        extracted = archive.extractfile(row.archive_path)

        if extracted is None:
            raise RuntimeError(row.archive_path)

        original = extracted.read().decode("utf-8-sig")
        cleaned = normalize_text(original)

        # Verify the transformation is exact and deterministic.
        assert cleaned == original.replace("\x0c", " ")
        assert "\x0c" not in cleaned

        original_words = len(word_pattern.findall(original))
        cleaned_words = len(word_pattern.findall(cleaned))

        normalization_records.append({
            "archive_path": row.archive_path,
            "formfeeds_replaced": original.count("\x0c"),
            "original_word_count": original_words,
            "cleaned_word_count": cleaned_words,
            "word_count_change": cleaned_words - original_words,
            "cleaned_sha256": hashlib.sha256(
                cleaned.encode("utf-8")
            ).hexdigest()
        })

normalization_df = pd.DataFrame(normalization_records)

# ------------------------------------------------------------
# 4. Verify one-to-one metadata alignment
# ------------------------------------------------------------

normalization_audit = speech_quality.merge(
    normalization_df,
    on="archive_path",
    how="left",
    validate="one_to_one",
    indicator=True
)

assert len(normalization_audit) == len(speech_quality)
assert normalization_audit["_merge"].eq("both").all()

normalization_audit.drop(columns="_merge", inplace=True)

# ------------------------------------------------------------
# 5. Summarize normalization results
# ------------------------------------------------------------

print("=" * 65)
print("CONTROLLED NORMALIZATION AUDIT")
print("=" * 65)

print(
    "Documents inspected:",
    len(normalization_audit)
)

print(
    "Documents modified:",
    int(normalization_audit["formfeeds_replaced"].gt(0).sum())
)

print(
    "Form-feed characters replaced:",
    int(normalization_audit["formfeeds_replaced"].sum())
)

print(
    "Documents with changed word counts:",
    int(normalization_audit["word_count_change"].ne(0).sum())
)

print("\nWord-count change distribution:")

print(
    normalization_audit["word_count_change"]
    .value_counts()
    .sort_index()
)

print("\nDocuments with largest word-count changes:")

display(
    normalization_audit.loc[
        normalization_audit["word_count_change"].ne(0),
        [
            "country_code",
            "year",
            "filename",
            "formfeeds_replaced",
            "original_word_count",
            "cleaned_word_count",
            "word_count_change"
        ]
    ].sort_values(
        "word_count_change",
        ascending=False
    ).head(20)
)



## 4. Historical Coverage and Speaker Metadata

### Objectives

This section evaluates the longitudinal coverage of the UNGDC corpus and inspects the supplementary speaker metadata.

The primary research comparison focuses on Iran and the United States. China is retained as a potential comparative case, while the complete corpus remains available for broader robustness analyses.

### Coverage assessment

- Identify available and missing country-year observations.
- Verify the distribution of speeches across the study period.
- Examine the coverage of Iran (IRN), the United States (USA), and China (CHN).
- Distinguish missing records from observed zero-valued measurements.
- Preserve historical country and entity identifiers without assuming that all identifiers represent politically continuous states.

### Speaker metadata assessment

The supplementary Excel file is inspected for its sheets, columns, missing values, and candidate linkage identifiers.

Speaker records will not be merged into the speech corpus until key uniqueness, temporal consistency, and matching coverage have been validated.

### Methodological safeguards

The unit of observation is the country/entity-year speech record. The presence of a record does not establish the completeness of the corresponding country's diplomatic activity.

Historical changes in state representation, including China's representation at the United Nations, require separate consideration before substantive interpretation.

No observations will be deleted, imputed, or merged during this stage.


In [ ]:

# ============================================================
# STEP 04: HISTORICAL COVERAGE AUDIT
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

FOCAL_COUNTRIES = {
    "IRN": "Iran",
    "USA": "United States",
    "CHN": "China",
}

STUDY_YEARS = list(range(1946, 2026))

# ------------------------------------------------------------
# 1. Create a complete country-year coverage grid
# ------------------------------------------------------------

coverage_grid = pd.MultiIndex.from_product(
    [list(FOCAL_COUNTRIES), STUDY_YEARS],
    names=["country_code", "year"]
).to_frame(index=False)

observed = (
    speech_quality[
        ["country_code", "year", "filename", "word_count"]
    ]
    .loc[
        lambda x: x["country_code"].isin(FOCAL_COUNTRIES)
    ]
    .copy()
)

assert not observed.duplicated(
    ["country_code", "year"]
).any(), "Duplicate country-year records detected."

coverage = coverage_grid.merge(
    observed,
    on=["country_code", "year"],
    how="left",
    validate="one_to_one",
    indicator=True
)

coverage["available"] = coverage["_merge"].eq("both")
coverage["country"] = coverage["country_code"].map(
    FOCAL_COUNTRIES
)

coverage = coverage.drop(columns="_merge")

# ------------------------------------------------------------
# 2. Summarize speech coverage
# ------------------------------------------------------------

coverage_summary = (
    coverage.groupby(["country_code", "country"])
    .agg(
        available_years=("available", "sum"),
        total_years=("year", "size"),
        first_available_year=(
            "year",
            lambda s: s[
                coverage.loc[s.index, "available"]
            ].min()
        ),
        last_available_year=(
            "year",
            lambda s: s[
                coverage.loc[s.index, "available"]
            ].max()
        )
    )
    .reset_index()
)

coverage_summary["missing_years"] = (
    coverage_summary["total_years"]
    - coverage_summary["available_years"]
)

coverage_summary["coverage_pct"] = (
    100
    * coverage_summary["available_years"]
    / coverage_summary["total_years"]
)

print("=" * 65)
print("HISTORICAL SPEECH COVERAGE")
print("=" * 65)

display(coverage_summary)

# ------------------------------------------------------------
# 3. Report missing years
# ------------------------------------------------------------

for code, name in FOCAL_COUNTRIES.items():

    missing = coverage.loc[
        (coverage["country_code"] == code)
        & (~coverage["available"]),
        "year"
    ].tolist()

    print(f"\n{name} ({code})")
    print(f"Missing years ({len(missing)}): {missing}")

# ------------------------------------------------------------
# 4. Count common observed years
# ------------------------------------------------------------

coverage_wide = coverage.pivot(
    index="year",
    columns="country_code",
    values="available"
)

print("\nOverlapping observed years:")

for left, right in [
    ("IRN", "USA"),
    ("CHN", "USA"),
    ("IRN", "CHN")
]:
    common = coverage_wide[left] & coverage_wide[right]

    print(
        f"{left} - {right}: "
        f"{int(common.sum())} years"
    )

# ------------------------------------------------------------
# 5. Visualize temporal availability
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 3.5))

for index, code in enumerate(FOCAL_COUNTRIES):

    available_years = coverage.loc[
        (coverage["country_code"] == code)
        & (coverage["available"]),
        "year"
    ]

    ax.scatter(
        available_years,
        [index] * len(available_years),
        marker="s",
        s=42,
        label=FOCAL_COUNTRIES[code]
    )

ax.set_yticks(range(len(FOCAL_COUNTRIES)))
ax.set_yticklabels(list(FOCAL_COUNTRIES.values()))

ax.set(
    title="UNGDC Speech Availability by Country (1946-2025)",
    xlabel="Year",
    xlim=(1945, 2026)
)

ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()



### 4.1. Speaker Metadata Inspection

#### Objective

This step evaluates the structure and completeness of the supplementary speaker metadata provided with the UN General Debate Corpus.

#### Procedures

1. Identify available Excel worksheets.
2. Inspect column names and data types.
3. Quantify missing values in each column.
4. Examine sample speaker records.
5. Identify potential country-year-session linkage variables.

#### Methodological considerations

Speaker metadata may contain incomplete information, especially for earlier historical periods.

The presence of multiple records for the same country-year combination will be assessed before merging.

No metadata records will be removed or modified during this inspection.


In [ ]:

# ============================================================
# STEP 04.1: SPEAKER METADATA INSPECTION
# ============================================================

excel_file = pd.ExcelFile(SPEAKERS_PATH)

print("=" * 65)
print("SPEAKER METADATA AUDIT")
print("=" * 65)

print("Available sheets:", excel_file.sheet_names)

speaker_sheets = {}

for sheet_name in excel_file.sheet_names:

    sheet_df = pd.read_excel(
        SPEAKERS_PATH,
        sheet_name=sheet_name
    )

    speaker_sheets[sheet_name] = sheet_df

    print("\n" + "-" * 65)
    print(f"Sheet: {sheet_name}")
    print(f"Rows: {len(sheet_df):,}")
    print(f"Columns: {len(sheet_df.columns)}")

    print("\nColumn names:")
    print(sheet_df.columns.tolist())

    print("\nData types:")
    display(
        sheet_df.dtypes
        .rename("dtype")
        .to_frame()
    )

    print("\nMissing values:")
    display(
        sheet_df.isna()
        .sum()
        .rename("missing_count")
        .to_frame()
    )

    print("\nSample records:")
    display(sheet_df.head(8))

print("\nMetadata inspection completed.")



### 4.2. Speaker Metadata Key Validation

#### Objective

This step evaluates whether speaker metadata can be reliably linked to the speech corpus using country, session, and year identifiers.

#### Validation procedures

1. Standardize column names without modifying source files.
2. Identify records with missing country identifiers.
3. Check the uniqueness of country-year-session keys.
4. Verify temporal consistency between session numbers and years.
5. Identify metadata records without corresponding speech files.
6. Identify speech files without corresponding speaker metadata.
7. Investigate non-empty values in auxiliary columns.

#### Methodological safeguards

The validation uses a composite key consisting of country code, year, and session.

Missing identifiers are not imputed automatically. Unmatched and duplicated records are retained for investigation.

No merge into the analytical master dataset will be performed until the linkage quality has been assessed.

Speaker metadata completeness and speech-text availability are evaluated separately.


In [ ]:

# ============================================================
# STEP 04.2: SPEAKER METADATA KEY VALIDATION
# ============================================================

# ------------------------------------------------------------
# 1. Load and standardize speaker metadata
# ------------------------------------------------------------

speakers_raw = speaker_sheets["Sheet1"].copy()

speaker_columns = {
    "Year": "year",
    "Session": "session",
    "ISO Code": "country_code",
    "Country": "country_name",
    "Name of Person Speaking": "speaker_name",
    "Post": "speaker_post"
}

speakers = speakers_raw.rename(
    columns=speaker_columns
).copy()

# Preserve all original rows for auditing.
speakers["source_row"] = speakers.index + 2

# Normalize whitespace in linkage identifiers.
speakers["country_code"] = (
    speakers["country_code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

speakers["country_code"] = (
    speakers["country_code"].replace("", pd.NA)
)

# ------------------------------------------------------------
# 2. Define the composite linkage key
# ------------------------------------------------------------

KEY = ["country_code", "year", "session"]

speech_keys = speech_index[KEY].drop_duplicates()

speaker_keys = speakers[KEY].copy()

# ------------------------------------------------------------
# 3. Identify missing and duplicated identifiers
# ------------------------------------------------------------

missing_key_rows = speakers[
    speakers[KEY].isna().any(axis=1)
].copy()

valid_speaker_keys = speakers[
    speakers[KEY].notna().all(axis=1)
].copy()

duplicate_speaker_keys = valid_speaker_keys[
    valid_speaker_keys.duplicated(
        subset=KEY,
        keep=False
    )
].sort_values(KEY)

# ------------------------------------------------------------
# 4. Check year-session consistency
# ------------------------------------------------------------

invalid_speaker_sessions = speakers[
    speakers["session"] != speakers["year"] - 1945
].copy()

# ------------------------------------------------------------
# 5. Compare speech and speaker identifiers
# ------------------------------------------------------------

key_comparison = speech_keys.merge(
    valid_speaker_keys[KEY].drop_duplicates(),
    on=KEY,
    how="outer",
    indicator=True,
    validate="one_to_one"
)

matched_keys = key_comparison[
    key_comparison["_merge"] == "both"
].copy()

speech_only = key_comparison[
    key_comparison["_merge"] == "left_only"
].copy()

speaker_only = key_comparison[
    key_comparison["_merge"] == "right_only"
].copy()

# ------------------------------------------------------------
# 6. Inspect auxiliary metadata
# ------------------------------------------------------------

auxiliary_column = "Unnamed: 6"

auxiliary_nonmissing = speakers_raw.loc[
    speakers_raw[auxiliary_column].notna(),
    ["Year", "Session", "ISO Code",
     "Country", auxiliary_column]
].copy()

# ------------------------------------------------------------
# 7. Report linkage quality
# ------------------------------------------------------------

print("=" * 65)
print("SPEAKER METADATA LINKAGE AUDIT")
print("=" * 65)

print(f"Speech records: {len(speech_index):,}")
print(f"Speaker metadata rows: {len(speakers):,}")

print(f"Missing-key metadata rows: {len(missing_key_rows):,}")
print(f"Duplicate-key metadata rows: {len(duplicate_speaker_keys):,}")
print(f"Invalid session-year rows: {len(invalid_speaker_sessions):,}")

print("-" * 65)

print(f"Matched speech keys: {len(matched_keys):,}")
print(f"Speech-only keys: {len(speech_only):,}")
print(f"Speaker-only keys: {len(speaker_only):,}")

match_rate = 100 * len(matched_keys) / len(speech_index)

print(f"Speech linkage coverage: {match_rate:.2f}%")

# ------------------------------------------------------------
# 8. Display records requiring investigation
# ------------------------------------------------------------

print("\nRecords with missing identifiers:")
display(missing_key_rows)

print("\nDuplicate speaker keys:")
display(duplicate_speaker_keys)

print("\nInvalid speaker session-year records:")
display(invalid_speaker_sessions)

print("\nSpeech records without speaker metadata:")
display(speech_only)

print("\nSpeaker metadata without matching speeches:")
display(speaker_only)

print("\nNon-empty auxiliary column values:")
display(auxiliary_nonmissing)



### 4.3. Investigation of Metadata Linkage Ambiguities

#### Objective

This step investigates duplicated speaker identifiers and unmatched records before selecting a metadata integration strategy.

#### Diagnostic approach

- Distinguish exact duplicate metadata records from records containing different speaker information.
- Quantify duplicate key groups rather than only duplicated rows.
- Examine unmatched observations by year and country.
- Inspect all linkage issues affecting Iran, the United States, and China.
- Preserve historical identifiers and all original metadata records.

#### Methodological decision

Multiple speakers associated with the same country-year-session key must not be arbitrarily reduced to a single speaker.

Any aggregation, deduplication, or historical identifier correction requires an explicit and reproducible decision rule.

The speech corpus remains the primary analytical population. Speaker metadata is supplementary and must not alter the number of speech observations.


In [ ]:

# ============================================================
# STEP 04.3: LINKAGE AMBIGUITY INVESTIGATION
# ============================================================

# ------------------------------------------------------------
# 1. Characterize duplicate key groups
# ------------------------------------------------------------

duplicate_groups = (
    duplicate_speaker_keys
    .groupby(KEY, dropna=False)
    .agg(
        n_rows=("source_row", "size"),
        n_speaker_names=("speaker_name", "nunique"),
        n_speaker_posts=("speaker_post", "nunique"),
        source_rows=("source_row", list),
        speaker_names=("speaker_name", list),
        speaker_posts=("speaker_post", list)
    )
    .reset_index()
)

print("=" * 65)
print("DUPLICATE SPEAKER KEY INVESTIGATION")
print("=" * 65)

print("Duplicate metadata rows:", len(duplicate_speaker_keys))
print("Duplicate key groups:", len(duplicate_groups))

print(
    "Groups with multiple distinct speaker names:",
    int(duplicate_groups["n_speaker_names"].gt(1).sum())
)

display(
    duplicate_groups.sort_values(
        ["n_speaker_names", "n_rows"],
        ascending=False
    )
)

# ------------------------------------------------------------
# 2. Identify exact duplicate metadata records
# ------------------------------------------------------------

identity_columns = [
    "country_code",
    "year",
    "session",
    "country_name",
    "speaker_name",
    "speaker_post"
]

exact_duplicate_rows = speakers[
    speakers.duplicated(
        subset=identity_columns,
        keep=False
    )
].sort_values(KEY)

print("\nExact duplicate metadata rows:")
print(len(exact_duplicate_rows))

display(
    exact_duplicate_rows[
        identity_columns + ["source_row"]
    ]
)

# ------------------------------------------------------------
# 3. Examine unmatched records by year
# ------------------------------------------------------------

print("\nSpeech-only records by year:")

display(
    speech_only.groupby("year")
    .size()
    .rename("n_unmatched_speeches")
    .sort_values(ascending=False)
    .head(20)
)

print("\nSpeaker-only records by year:")

display(
    speaker_only.groupby("year")
    .size()
    .rename("n_unmatched_metadata_keys")
    .sort_values(ascending=False)
    .head(20)
)

# ------------------------------------------------------------
# 4. Examine the focal countries
# ------------------------------------------------------------

FOCAL_CODES = ["IRN", "USA", "CHN"]

focal_speech_only = speech_only[
    speech_only["country_code"].isin(FOCAL_CODES)
]

focal_speaker_only = speaker_only[
    speaker_only["country_code"].isin(FOCAL_CODES)
]

focal_duplicate_groups = duplicate_groups[
    duplicate_groups["country_code"].isin(FOCAL_CODES)
]

print("\n" + "=" * 65)
print("FOCAL COUNTRY LINKAGE DIAGNOSTICS")
print("=" * 65)

print("Unmatched focal speech records:")
display(focal_speech_only.sort_values(KEY))

print("\nUnmatched focal metadata keys:")
display(focal_speaker_only.sort_values(KEY))

print("\nDuplicated focal speaker keys:")
display(focal_duplicate_groups.sort_values(KEY))

# ------------------------------------------------------------
# 5. Verify metadata audit accounting
# ------------------------------------------------------------

assert (
    len(matched_keys) + len(speech_only)
    == len(speech_index)
), "Speech key accounting failed."

assert (
    len(matched_keys) + len(speaker_only)
    == len(valid_speaker_keys[KEY].drop_duplicates())
), "Speaker key accounting failed."

print("\nLinkage accounting checks: PASSED")



### 4.4. Construction of the Three-Country Analytical Cohort

#### Objective

Construct a focused analytical cohort comprising Iran (IRN), the United States (USA), and China (CHN), while preserving the complete original UNGDC archive.

#### Cohort definition

- Iran (IRN): 74 available speeches.
- United States (USA): 80 available speeches.
- China (CHN): 79 available speeches.
- Expected analytical cohort: 233 speech records.

#### Metadata integration

Speaker metadata is linked using the composite country-year-session identifier.

A conservative aggregation strategy preserves all associated speaker records without arbitrarily assigning a primary speaker.

#### Methodological safeguards

1. The raw corpus remains unchanged.
2. Country selection is explicitly documented.
3. No selected speech is removed because of missing speaker metadata.
4. Multiple speaker assignments are retained and flagged.
5. The speech-level dataset must contain exactly one row per speech.
6. Unmatched speaker information remains missing.
7. Historical changes in China's UN representation must be considered in longitudinal interpretation.

#### Scope

The three-country cohort is intended for the primary comparative analysis.

The full UNGDC corpus remains available for future reference-country comparisons, alternative topic-model training, and robustness analyses.


In [ ]:

# ============================================================
# STEP 04.4: THREE-COUNTRY ANALYTICAL COHORT
# ============================================================

# ------------------------------------------------------------
# 1. Define the analytical cohort
# ------------------------------------------------------------

FOCAL_CODES = ["IRN", "USA", "CHN"]
KEY = ["country_code", "year", "session"]

speech_cohort = speech_quality.loc[
    speech_quality["country_code"].isin(FOCAL_CODES)
].copy()

# Validate expected cohort composition.
expected_counts = {
    "IRN": 74,
    "USA": 80,
    "CHN": 79
}

actual_counts = (
    speech_cohort["country_code"]
    .value_counts()
    .to_dict()
)

assert actual_counts == expected_counts, (
    f"Unexpected cohort composition: {actual_counts}"
)

assert len(speech_cohort) == 233
assert speech_cohort["archive_path"].is_unique

# ------------------------------------------------------------
# 2. Select speaker metadata for focal countries
# ------------------------------------------------------------

speaker_cohort = valid_speaker_keys.loc[
    valid_speaker_keys["country_code"].isin(FOCAL_CODES)
].copy()

speaker_cohort = speaker_cohort.sort_values(
    KEY + ["source_row"]
)

# ------------------------------------------------------------
# 3. Preserve all available speaker records
# ------------------------------------------------------------

def preserve_values(series):
    """Preserve original non-missing values in source order."""
    return [
        str(value)
        for value in series
        if pd.notna(value)
    ]


speaker_aggregated = (
    speaker_cohort
    .groupby(KEY, as_index=False)
    .agg(
        metadata_row_count=("source_row", "size"),
        speaker_names=("speaker_name", preserve_values),
        speaker_posts=("speaker_post", preserve_values),
        source_rows=("source_row", list)
    )
)

speaker_aggregated["speaker_assignment_ambiguous"] = (
    speaker_aggregated["metadata_row_count"] > 1
)

assert not speaker_aggregated.duplicated(KEY).any()

# ------------------------------------------------------------
# 4. Perform a validated speech-level linkage
# ------------------------------------------------------------

speech_master = speech_cohort.merge(
    speaker_aggregated,
    on=KEY,
    how="left",
    validate="one_to_one",
    indicator="speaker_merge_status"
)

speech_master["has_speaker_metadata"] = (
    speech_master["speaker_merge_status"] == "both"
)

speech_master["speaker_assignment_ambiguous"] = (
    speech_master["speaker_assignment_ambiguous"]
    .fillna(False)
    .astype(bool)
)

# ------------------------------------------------------------
# 5. Validate speech population preservation
# ------------------------------------------------------------

assert len(speech_master) == 233
assert speech_master["archive_path"].is_unique

assert set(speech_master["archive_path"]) == set(
    speech_cohort["archive_path"]
)

assert speech_master[KEY].notna().all().all()

assert not speech_master.duplicated(KEY).any()

# ------------------------------------------------------------
# 6. Produce cohort-level quality summary
# ------------------------------------------------------------

cohort_summary = (
    speech_master
    .groupby("country_code")
    .agg(
        n_speeches=("archive_path", "size"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        metadata_matched=("has_speaker_metadata", "sum"),
        ambiguous_speakers=(
            "speaker_assignment_ambiguous", "sum"
        ),
        mean_words=("word_count", "mean"),
        median_words=("word_count", "median")
    )
    .reset_index()
)

cohort_summary["metadata_coverage_pct"] = (
    100
    * cohort_summary["metadata_matched"]
    / cohort_summary["n_speeches"]
)

# ------------------------------------------------------------
# 7. Report validation results
# ------------------------------------------------------------

print("=" * 65)
print("THREE-COUNTRY ANALYTICAL COHORT")
print("=" * 65)

print(f"Total speeches: {len(speech_master):,}")
print(f"Countries: {', '.join(FOCAL_CODES)}")

print(
    "Matched speaker metadata:",
    int(speech_master["has_speaker_metadata"].sum())
)

print(
    "Unmatched speaker metadata:",
    int((~speech_master["has_speaker_metadata"]).sum())
)

print(
    "Ambiguous speaker assignments:",
    int(speech_master["speaker_assignment_ambiguous"].sum())
)

print("\nCountry-level cohort summary:")
display(cohort_summary.round(2))

print("\nUnmatched speech records:")
display(
    speech_master.loc[
        ~speech_master["has_speaker_metadata"],
        KEY + ["filename"]
    ]
)

print("\nCohort construction: PASSED")



## 5. Master Text Dataset Construction

### Objective

Construct a speech-level analytical dataset containing the full texts of Iran, the United States, and China for the period 1946–2025.

### Text representations

- `text_raw`: Original decoded speech content, without linguistic preprocessing.
- `text_clean`: Minimally normalized text, replacing form-feed characters (U+000C) with spaces.

### Data integrity requirements

1. Preserve all 233 selected speeches.
2. Preserve the original speech identifiers and metadata.
3. Retain speaker metadata even when incomplete.
4. Verify each source document against its previously computed SHA-256 hash.
5. Confirm that normalization modifies only form-feed characters.
6. Verify that original and cleaned word counts match the earlier audit.
7. Ensure that all speech texts remain non-empty.

### Reproducibility

The compressed source archive remains unchanged. Original document bytes are verifiable through the stored SHA-256 hashes.

The `text_raw` representation is decoded using UTF-8 with optional BOM removal. This representation is not a byte-for-byte copy of the original file.

No documents are excluded, truncated, or linguistically transformed in this step.


In [ ]:

# ============================================================
# STEP 05: MASTER TEXT DATASET CONSTRUCTION
# ============================================================

import hashlib
import tarfile

# ------------------------------------------------------------
# 1. Load full texts from the original archive
# ------------------------------------------------------------

text_records = []

with tarfile.open(ARCHIVE_PATH, mode="r:gz") as archive:

    for row in speech_master.itertuples(index=False):

        extracted = archive.extractfile(row.archive_path)

        if extracted is None:
            raise RuntimeError(
                f"Unable to extract: {row.archive_path}"
            )

        raw_bytes = extracted.read()

        # Verify source integrity.
        current_hash = hashlib.sha256(
            raw_bytes
        ).hexdigest()

        assert current_hash == row.sha256, (
            f"Source hash mismatch: {row.filename}"
        )

        original_text = raw_bytes.decode("utf-8-sig")
        cleaned_text = normalize_text(original_text)

        # Verify that only the intended transformation occurred.
        assert cleaned_text == original_text.replace(
            "\x0c", " "
        )

        assert "\x0c" not in cleaned_text
        assert original_text.strip()
        assert cleaned_text.strip()

        text_records.append({
            "archive_path": row.archive_path,
            "text_raw": original_text,
            "text_clean": cleaned_text
        })

# ------------------------------------------------------------
# 2. Build the text table
# ------------------------------------------------------------

text_df = pd.DataFrame(text_records)

assert len(text_df) == 233
assert text_df["archive_path"].is_unique

# ------------------------------------------------------------
# 3. Integrate texts with the master metadata
# ------------------------------------------------------------

speech_master = speech_master.merge(
    text_df,
    on="archive_path",
    how="left",
    validate="one_to_one",
    indicator="text_merge_status"
)

assert speech_master["text_merge_status"].eq("both").all()

speech_master = speech_master.drop(
    columns="text_merge_status"
)

# ------------------------------------------------------------
# 4. Validate document-level text integrity
# ------------------------------------------------------------

speech_master["raw_word_count_check"] = (
    speech_master["text_raw"].apply(
        lambda text: len(word_pattern.findall(text))
    )
)

speech_master["clean_word_count_check"] = (
    speech_master["text_clean"].apply(
        lambda text: len(word_pattern.findall(text))
    )
)

assert (
    speech_master["raw_word_count_check"]
    == speech_master["word_count"]
).all(), "Raw word-count validation failed."

assert (
    speech_master["raw_word_count_check"]
    == speech_master["clean_word_count_check"]
).all(), "Normalization changed word counts."

assert speech_master["text_raw"].notna().all()
assert speech_master["text_clean"].notna().all()

assert speech_master["archive_path"].is_unique
assert len(speech_master) == 233

# ------------------------------------------------------------
# 5. Report the master dataset validation
# ------------------------------------------------------------

print("=" * 65)
print("MASTER TEXT DATASET VALIDATION")
print("=" * 65)

print(f"Total speeches: {len(speech_master):,}")

print(
    "Countries:",
    sorted(speech_master["country_code"].unique())
)

print(
    "Raw texts available:",
    int(speech_master["text_raw"].notna().sum())
)

print(
    "Cleaned texts available:",
    int(speech_master["text_clean"].notna().sum())
)

print(
    "Documents modified by normalization:",
    int(
        (
            speech_master["text_raw"]
            != speech_master["text_clean"]
        ).sum()
    )
)

print(
    "Raw/clean word-count mismatches:",
    int(
        (
            speech_master["raw_word_count_check"]
            != speech_master["clean_word_count_check"]
        ).sum()
    )
)

print("\nSpeech counts by country:")

display(
    speech_master.groupby("country_code")
    .size()
    .rename("n_speeches")
    .to_frame()
)

print("\nSample records:")

display(
    speech_master[
        [
            "country_code",
            "year",
            "filename",
            "word_count",
            "has_speaker_metadata",
            "utf8_valid"
        ]
    ].head(10)
)

print("\nMASTER TEXT DATASET: VALIDATION PASSED")



### 5.1. Master Dataset Export

#### Objective

Export the validated three-country speech dataset into a reusable Apache Parquet file for subsequent exploratory, NLP, and statistical analyses.

#### Output

`data/processed/speeches_master.parquet`

#### Dataset characteristics

- Unit of observation: Country-year speech.
- Study period: 1946–2025.
- Countries: Iran, United States, and China.
- Expected observations: 233.
- Text representations: Original decoded text and minimally normalized text.
- Speaker metadata: Preserved with missingness and ambiguity indicators.
- Quality indicators: Document length, encoding status, and content hashes.

#### Reproducibility

The exported dataset serves as the common input for the remaining analysis notebooks.

The original source files remain unchanged.

Export integrity will be evaluated through an independent read-back validation step.


In [ ]:

# ============================================================
# STEP 05.1: EXPORT MASTER DATASET
# ============================================================

import json

OUTPUT_PATH = (
    PROCESSED_DIR / "speeches_master.parquet"
)

# Preserve the validated in-memory dataset.
export_df = speech_master.copy()

# Serialize list-valued metadata for stable Parquet storage.
list_columns = [
    "speaker_names",
    "speaker_posts",
    "source_rows"
]

for column in list_columns:
    export_df[column] = export_df[column].apply(
        lambda value: (
            json.dumps(value, ensure_ascii=False)
            if isinstance(value, list)
            else None
        )
    )

# Store the validated speech-level dataset.
# export_df.to_parquet(
#     OUTPUT_PATH,
#     engine="pyarrow",
#     compression="zstd",
#     index=False
# )

print("=" * 65)
print("MASTER DATASET EXPORT")
print("=" * 65)

print(f"Output: {OUTPUT_PATH}")
print(f"Records exported: {len(export_df):,}")

print(
    "File size:",
    f"{OUTPUT_PATH.stat().st_size / (1024 ** 2):.2f} MB"
)

print("Export completed.")


### Python: Validate Saved Dataset

In [ ]:

# ============================================================
# STEP 05.2: INDEPENDENT PARQUET VALIDATION
# ============================================================

import pandas as pd
import json

# Read the dataset independently from disk.
reloaded = pd.read_parquet(
    OUTPUT_PATH,
    engine="pyarrow"
)

# ------------------------------------------------------------
# 1. Validate structure and identifiers
# ------------------------------------------------------------

assert len(reloaded) == 233
assert reloaded["archive_path"].is_unique

assert set(reloaded["country_code"]) == {
    "IRN", "USA", "CHN"
}

assert reloaded["text_raw"].notna().all()
assert reloaded["text_clean"].notna().all()

# ------------------------------------------------------------
# 2. Validate complete text preservation
# ------------------------------------------------------------

original_indexed = speech_master.set_index(
    "archive_path"
).sort_index()

reloaded_indexed = reloaded.set_index(
    "archive_path"
).sort_index()

assert original_indexed.index.equals(
    reloaded_indexed.index
)

assert (
    original_indexed["text_raw"]
    == reloaded_indexed["text_raw"]
).all()

assert (
    original_indexed["text_clean"]
    == reloaded_indexed["text_clean"]
).all()

# ------------------------------------------------------------
# 3. Validate serialized metadata
# ------------------------------------------------------------

for column in [
    "speaker_names",
    "speaker_posts",
    "source_rows"
]:
    restored = reloaded[column].apply(
        lambda value: (
            json.loads(value)
            if isinstance(value, str)
            else []
        )
    )

    assert restored.apply(
        lambda value: isinstance(value, list)
    ).all()

# ------------------------------------------------------------
# 4. Validate country-level counts
# ------------------------------------------------------------

expected_counts = {
    "IRN": 74,
    "USA": 80,
    "CHN": 79
}

actual_counts = (
    reloaded["country_code"].value_counts().to_dict()
)

assert actual_counts == expected_counts

# ------------------------------------------------------------
# 5. Report validation results
# ------------------------------------------------------------

print("=" * 65)
print("PARQUET READ-BACK VALIDATION")
print("=" * 65)

print("Total records:", len(reloaded))
print("Unique speeches:", reloaded["archive_path"].nunique())
print("Original texts preserved: True")
print("Cleaned texts preserved: True")
print("Speaker metadata decoding: PASSED")

print("\nCountry composition:")
display(
    reloaded["country_code"]
    .value_counts()
    .rename("n_speeches")
    .to_frame()
)

print("\nPARQUET VALIDATION: PASSED")
